In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import riskfolio as rp

data = pd.read_csv('data/historical_data.csv').drop(columns=['Unnamed: 0'])

In [13]:
prices = data.pivot(index='timestamp', columns='ticker', values='close')
prices

ticker,BND,SPY,VNLA,VNQ,VT
timestamp,,,,,
2023-12-19 05:00:00+00:00,73.39,474.84,48.35,88.75,102.17
2023-12-20 05:00:00+00:00,73.63,468.26,48.39,87.67,100.77
2023-12-21 05:00:00+00:00,73.62,472.70,48.44,87.37,102.07
2023-12-22 05:00:00+00:00,73.36,473.65,48.25,87.71,102.22
2023-12-26 05:00:00+00:00,73.43,475.65,48.28,88.36,102.73
...,...,...,...,...,...
2025-12-11 05:00:00+00:00,74.26,689.17,49.20,89.56,142.56
2025-12-12 05:00:00+00:00,74.03,681.76,49.21,89.45,141.22
2025-12-15 05:00:00+00:00,74.12,680.73,49.22,89.73,141.22


In [14]:
returns = prices.pct_change().dropna()

In [15]:
scenarios = {
    'Conservative': {'SPY': 0.20, 'VNQ': 0.10, 'BND': 0.50, 'VT': 0.10, 'VNLA': 0.10},
    'Balanced': {'SPY': 0.40, 'VNQ': 0.15, 'BND': 0.25, 'VT': 0.15, 'VNLA': 0.05},
    'Aggressive': {'SPY': 0.60, 'VNQ': 0.20, 'BND': 0.10, 'VT': 0.10, 'VNLA': 0.00}
}


portfolio_value = 100000000
confidence_levels = [0.95, 0.99]

In [16]:
def calculate_portfolio_risk_metrics(returns_df, weights_dict, portfolio_val, conf_levels):
    
    weights = np.array([weights_dict[col] for col in returns_df.columns])
    
    portfolio_returns = returns_df @ weights
    
    results = {}
    
    for conf in conf_levels:
        alpha = 1 - conf 
        
        var_return = rp.VaR_Hist(portfolio_returns.values.reshape(-1, 1), alpha=alpha)
        var_dollar = abs(var_return) * portfolio_val
        
        cvar_return = rp.CVaR_Hist(portfolio_returns.values.reshape(-1, 1), alpha=alpha)
        cvar_dollar = abs(cvar_return) * portfolio_val
        
        results[f'{int(conf*100)}%'] = {
            'VaR (%)': abs(var_return) * 100,
            'VaR ($)': var_dollar,
            'CVaR (%)': abs(cvar_return) * 100,
            'CVaR ($)': cvar_dollar,
            'portfolio_returns': portfolio_returns 
        }
    
    return results


In [17]:
all_results = {}
for scenario_name, weights in scenarios.items():
    all_results[scenario_name] = calculate_portfolio_risk_metrics(
        returns, weights, portfolio_value, confidence_levels
    )

In [18]:
def create_summary_table(results_dict, portfolio_val):
    """Generate executive summary of risk metrics"""
    
    summary_data = []
    
    for scenario, metrics in results_dict.items():
        for conf_level in ['95%', '99%']:
            summary_data.append({
                'Scenario': scenario,
                'Confidence': conf_level,
                'VaR ($)': f"${metrics[conf_level]['VaR ($)']:,.0f}",
                'VaR (%)': f"{metrics[conf_level]['VaR (%)']:.2f}%",
                'CVaR ($)': f"${metrics[conf_level]['CVaR ($)']:,.0f}",
                'CVaR (%)': f"{metrics[conf_level]['CVaR (%)']:.2f}%"
            })
    
    df = pd.DataFrame(summary_data)
    return df

summary_df = create_summary_table(all_results, portfolio_value)
print(f"\n📈 Portfolio Risk Analysis (${portfolio_value:,} Portfolio)\n")
print(summary_df.to_string(index=False))


📈 Portfolio Risk Analysis ($100,000,000 Portfolio)

    Scenario Confidence    VaR ($) VaR (%)   CVaR ($) CVaR (%)
Conservative        95%   $620,752   0.62% $1,025,284    1.03%
Conservative        99% $1,194,786   1.19% $1,688,879    1.69%
    Balanced        95%   $912,181   0.91% $1,562,380    1.56%
    Balanced        99% $1,633,945   1.63% $2,801,557    2.80%
  Aggressive        95% $1,204,514   1.20% $1,992,388    1.99%
  Aggressive        99% $2,096,044   2.10% $3,596,140    3.60%


In [19]:
def create_loss_distribution_plot(results_dict, scenarios_dict):    
    # Prepare data for plotting
    plot_data = []
    for scenario_name in scenarios_dict.keys():
        port_returns = all_results[scenario_name]['95%']['portfolio_returns']
        losses_pct = -port_returns * 100  # Convert to % losses
        
        for loss in losses_pct:
            plot_data.append({
                'Scenario': scenario_name,
                'Loss (%)': loss
            })
    
    df_plot = pd.DataFrame(plot_data)
    
    fig = px.histogram(
        df_plot, 
        x='Loss (%)', 
        color='Scenario',
        facet_col='Scenario',
        facet_col_wrap=3,
        title='Historical Loss Distributions with VaR and CVaR Thresholds',
        nbins=50,
        opacity=0.75,
        height=500
    )
    
    for i, scenario_name in enumerate(scenarios_dict.keys()):
        var_95 = all_results[scenario_name]['95%']['VaR (%)']
        cvar_95 = all_results[scenario_name]['95%']['CVaR (%)']
        
        fig.add_vline(
            x=var_95, 
            line_dash="dash", 
            line_color="red",
            annotation_text=f"VaR: {var_95:.2f}%",
            annotation_position="left",
            row=1, col=i+1
        )
        
        fig.add_vline(
            x=cvar_95, 
            line_dash="solid", 
            line_color="darkred",
            annotation_text=f"CVaR: {cvar_95:.2f}%",
            annotation_position="right",
            row=1, col=i+1
        )
    
    fig.update_xaxes(title_text="Portfolio Loss (%)")
    fig.update_yaxes(title_text="Frequency")
    fig.update_layout(showlegend=False)
    
    return fig

fig = create_loss_distribution_plot(all_results, scenarios)
fig.show()
